# Plan 004.9 — Duplicate Pagination

**Audience:** Python and CLI users reviewing duplicate groups iteratively.
**Prerequisites:** basic Catalog scans and duplicate groups.
**Learning goals:** inspect groups 11–20 and 91–100, distinguish offsets from
member limits, and understand live pages versus scan-pinned streams and handles.

1. Build 100 duplicate groups from disposable synthetic files.
2. Fetch pages and stream suffixes; inspect the same CLI ranges.
3. Refresh between requests and retain an old handle safely.
4. Practice final-page selection.

API offsets are zero-based; displayed group numbers are one-based. All file
creation and refresh demonstrations below occur inside one temporary directory.
See [the plan](../../docs/plans/004.9-duplicate-pagination.md),
[query grammar](../../docs/query-grammar.md#duplicate-pagination), and
[CLI reference](../../docs/catalog-cli.md#duplicate-pages).

## 1. Create a disposable catalog

Each identity has two copies, including empty content. Distinct content sizes
make the default redundant-bytes descending order easy to recognize.

In [ ]:
from contextlib import closing, redirect_stdout
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog
from archiver.cli import main

temporary = TemporaryDirectory(prefix="archiver-pagination-")
root = Path(temporary.name) / "source"
root.mkdir()
for index in range(100):
    directory = root / "nested" / f"group-{index:03}"
    directory.mkdir(parents=True)
    for member in range(2):
        (directory / f"copy-{member}.bin").write_bytes(bytes([index]) * index)
with redirect_stdout(StringIO()):
    assert main(["catalog", "init", str(root)]) == 0
catalog = Catalog.open(root / ".archiver" / "catalog.sqlite")
catalog.scan_directory(root, excluded_directories=(root / ".archiver",))
query = catalog.duplicates(root)
assert query.summary().duplicate_content_group_count == 100
print("100 duplicate groups, 200 file instances; empty content included.")

## 2. Fetch groups 11–20 and 91–100

`offset(10).limit(10)` means positions `[10, 20)`. Each fetch is bounded in SQL.
The summary still describes all 100 groups; member limits are independent.

In [ ]:
def source_snapshot():
    return {
        path.relative_to(root).as_posix(): (path.read_bytes(), path.stat().st_mtime_ns, path.stat().st_mode)
        for path in root.rglob("*.bin")
    }

before_reads = source_snapshot()
for offset in (10, 90):
    page_query = query.offset(offset).limit(10).member_limit(1)
    rows = page_query.fetch()
    assert len(rows) == 10
    assert page_query.summary() == query.summary()
    print(f"Groups {offset + 1}–{offset + len(rows)} of 100")
    for position, group in enumerate(rows, start=offset + 1):
        assert len(group.members) == 1 and group.file_instance_count == 2
        print(position, group.content_id.digest[:12], group.size_bytes, "bytes")
assert len(query.offset(97).limit(10).fetch()) == 3
assert query.offset(100).fetch() == ()
assert query.group_offset == 0  # Transformations leave the base reusable.

## 3. Stream a bounded page or the remaining suffix

`stream_all()` **ignores `.limit()` but keeps `.offset()`**. With 100 groups,
`offset(90)` skips the first 90, leaving 10:

```python
query.offset(90).limit(2).fetch()       # 2 groups: positions 90–91, as a tuple
query.offset(90).limit(2).stream()      # 2 groups: positions 90–91, as an iterator
query.offset(90).limit(2).stream_all()  # 10 groups: positions 90–99, as an iterator
```

Use `fetch()` or `stream()` when you want the limit to apply. Reset the offset
to zero before `stream_all()` to process all 100 groups. Close streams on early exits; the catalog
must remain open. Group offsets never skip members of a selected group.

In [ ]:
assert len(query.offset(90).limit(2).fetch()) == 2
with closing(query.offset(90).limit(2).stream()) as groups:
    assert sum(1 for _ in groups) == 2
with closing(query.offset(90).limit(2).stream_all()) as groups:
    assert sum(1 for _ in groups) == 10
with closing(query.offset(90).offset(0).stream_all()) as groups:
    assert sum(1 for _ in groups) == 100
print("Bounded page: 2 groups; suffix: 10 groups; reset: all 100.")

## 4. Inspect the same ranges through the CLI

These are real CLI parser/execution calls against the disposable catalog. The
command equivalents are `uv run archiver catalog duplicates ROOT --details
--group-offset 10 --group-limit 10 --member-limit 1` and offset 90. The GROUP
column and footer use absolute one-based positions. Here we print the footer
and check the full output without saving a large table.

In [ ]:
for offset in (10, 90):
    captured = StringIO()
    with redirect_stdout(captured):
        assert main([
            "catalog", "duplicates", str(root), "--details",
            "--group-offset", str(offset), "--group-limit", "10", "--member-limit", "1",
        ]) == 0
    output = captured.getvalue()
    expected_footer = f"Showing groups {offset + 1}-{offset + 10} of 100 duplicate groups."
    assert expected_footer in output
    print(expected_footer)
assert source_snapshot() == before_reads  # All page and CLI reads left sources unchanged.

## 5. A refresh can shift the next page

Each fetch resolves the latest successful scan. A stream resolves once when
first advanced. Inserting an earlier-ranked group can repeat a group across
successive page requests; removing one can skip a group. Row numbers are not
content identity. Failed scans do not become current.

We insert a new largest group explicitly, within the disposable source. The
already started stream will retain its old sequence, while the deferred stream
and fresh page use the refreshed ordering.

In [ ]:
before_page = query.offset(10).limit(10).fetch()
first_page_ids = [group.content_id for group in query.limit(10).fetch()]
deferred = query.offset(10).limit(10).stream()
with closing(query.offset(10).limit(10).stream()) as started:
    retained = next(started)
    assert retained.content_id == before_page[0].content_id
    added = root / "new-largest"
    added.mkdir()
    for member in range(2):
        (added / f"copy-{member}.bin").write_bytes(b"x" * 101)
    catalog.scan_directory(root, excluded_directories=(root / ".archiver",))
    fresh_page = query.offset(10).limit(10).fetch()
    assert fresh_page[0].content_id == first_page_ids[-1]  # A repeat after the rank shift.
    with closing(deferred):
        assert [group.content_id for group in deferred] == [group.content_id for group in fresh_page]
    assert [group.content_id for group in started] == [group.content_id for group in before_page[1:]]
print("Fresh pages shifted after refresh; the started stream stayed on its original scan.")

## 6. A retained handle keeps its observed members

Closing the outer stream does not detach its handles. We add a third copy of
that content and refresh again. The retained handle still exposes two observed
members; a fresh group terminal exposes three. This pins catalog observations,
not the live bytes on disk.

CLI summary and row fetch are also separate live terminals: their totals need
not share one scan under concurrent refresh. Avoid successful refreshes during
a review that requires stable page positions; no paging session or scan-selector
API is introduced here.

In [ ]:
with closing(retained.stream_all_members()) as members:
    old_members = tuple(members)
assert len(old_members) == 2
original_path = root / old_members[0].relative_path
(original_path.parent / "copy-extra.bin").write_bytes(original_path.read_bytes())
catalog.scan_directory(root, excluded_directories=(root / ".archiver",))
with closing(retained.stream_all_members()) as members:
    assert len(tuple(members)) == 2
with closing(query.stream_all()) as groups:
    current = next(group for group in groups if group.content_id == retained.content_id)
    assert current.file_instance_count == 3
print("Retained handle: 2 members; fresh group: 3 members.")

## 7. Exercise: final five groups

Derive a query for the last five groups in the current state. Keep its complete
summary unchanged and reset its offset to zero afterward. Try another sort order
as an extension; changing order changes what page positions mean.

In [ ]:
total = query.summary().duplicate_content_group_count
last_five = query.offset(max(0, total - 5)).limit(5)
assert len(last_five.fetch()) == 5
assert last_five.summary() == query.summary()
assert last_five.offset(0).group_offset == 0
print(f"Last page: groups {total - 4}–{total}.")

## Cleanup and pitfalls

Offsets must be exact nonnegative integers within SQLite's signed-integer range;
limits remain positive. `stream_all()` keeps the offset, summaries ignore it,
and group numbers are temporary positions. This feature does not choose keepers,
mutate source files during reads, or manage archive storage.

In [ ]:
catalog.close()
temporary.cleanup()
print("Disposable catalog and source removed.")